# RNA3D — Pilot10 native-chain provenance & residue mapping audit (CPU only)

**Purpose:** Check each frozen RNA chain directly against its archived **original mmCIF**, its frozen FASTA, per-residue map CSV, and native evaluation PDB. No inference or GPU required. **No GitHub writes.** This is **not** training-overlap clearance or a full RNA benchmark validation.

**Safety:** Read-only input verification; writes only to a *new*, uniquely named directory under `MyDrive/RNA3D_Benchmark/Evaluations`. If the directory already exists the notebook **stops**, never overwrites. Review all `FAIL`/`REVIEW` flags before inferring validity. Do not rerun completed prior inference/evaluation notebooks.

**Critical scientific questions:** (1) Does requested chain map to a unique mmCIF polymer? (2) Does its complete polymer sequence match FASTA? (3) Do observed positions, residue names, and author numbering agree with raw coordinate records? (4) Does our native evaluation PDB reproduce raw experimental **C3′** positions? (5) Are release dates available for future leakage checks?

## 1. Mount Google Drive (CPU runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install minimal parser (Gemmi)

In [ ]:
%pip -q install 'gemmi==0.7.4'
import gemmi
print('Gemmi:',gemmi.__version__)

## 3. Paths and immutable pilot target set

In [ ]:
from pathlib import Path
import csv, json, hashlib, math, re, os, shutil, statistics
from datetime import datetime, timezone
from collections import defaultdict, Counter

ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
CIFS=ROOT/'native/pilot_v1/raw_cif'
NATIVE=ROOT/'native/pilot_v1/native_eval'
NUFOLD=ROOT/'NuFold/runs/pilot10_single_20261018T203832Z'
TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B',
         '9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
AUDIT_ID='pilot10_native_chain_provenance_20261113_v1'
DEST=ROOT/'Evaluations'/AUDIT_ID
STAGE=Path('/content/rna3d_pilot10_native_chain_audit_stage')
assert len(TARGETS)==len(set(TARGETS))==10
for x in (ROOT,CIFS,NATIVE,NUFOLD,ROOT/'Evaluations'):
    assert x.is_dir(),f'Missing prerequisite Drive folder: {x}'
assert not DEST.exists(),f'Audit folder already exists: {DEST}. Inspect it; DO NOT overwrite.'
assert not STAGE.exists(),f'Local staging exists: {STAGE}. Inspect before retrying.'
print('Frozen targets:',', '.join(TARGETS))
print('Raw mmCIF:',CIFS)
print('Destination:',DEST)
print('CPU-only. Inference and prior experiment folders are read-only.')

## 4. Parsing, identity and coordinate checks

In [ ]:
NT_CODES={'A':'A','C':'C','G':'G','U':'U','RA':'A','RC':'C','RG':'G','RU':'U',
          'ADE':'A','CYT':'C','GUA':'G','URA':'U','PSU':'U'}

def hsh(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def read_fasta(p):
    x=[s.strip() for s in p.read_text().splitlines() if s.strip()]
    assert x and x[0].startswith('>') and sum(s.startswith('>') for s in x)==1
    seq=''.join(s for s in x if not s.startswith('>')).upper().replace('T','U')
    assert seq and set(seq)<=set('ACGU')
    return seq

def rows(path):
    with path.open(newline='') as f: return list(csv.DictReader(f))

def cif_records(block,category):
    t=block.find_mmcif_category(category)
    if not len(t):return []
    keys=[tag.split('.',1)[-1] for tag in t.tags]
    return [dict(zip(keys,r)) for r in t]

def null(v):return v is None or str(v).strip() in ('?','.','')

def clean(v):
    if null(v):return ''
    s=str(v).strip()
    return s[1:-1] if len(s)>=2 and s[0]==s[-1] and s[0] in ("'", '"') else s

def number_equal(a,b):
    a=clean(a);b=clean(b)
    if a==b:return True
    try:return float(a)==float(b)
    except (ValueError,TypeError):return False

def atom_coords_pdb(path,atom="C3'"):
    result={}; mult=defaultdict(list)
    for line in path.read_text(errors='replace').splitlines():
        if not line.startswith(('ATOM  ','HETATM')) or len(line)<54:continue
        if line[12:16].strip()!=atom:continue
        if line[16] not in (' ','A','1'):continue
        try:
            pos=int(line[22:26]); xyz=tuple(float(line[z:z+8]) for z in (30,38,46))
        except ValueError:continue
        mult[pos].append(xyz)
    return {p:v[0] for p,v in mult.items()}, {p:len(v) for p,v in mult.items() if len(v)>1}

def distance(a,b):return math.sqrt(sum((x-y)**2 for x,y in zip(a,b)))

def first(block,tag):
    value=block.find_value(tag)
    return clean(value) if value else ''

# Parser sanity checks
assert NT_CODES['PSU']=='U'
assert number_equal('117','117') and not number_equal('117','118')
print('Helper tests PASS')

## 5. Audit all ten original mmCIF references

**Identity policy:** Resolve the target chain from `_pdbx_poly_seq_scheme.pdb_strand_id` (author chain) and validate it against `_atom_site.auth_asym_id`; confirm the polymer's label asymmetric unit via `_pdbx_poly_seq_scheme.asym_id`. Then require the entire `_pdbx_poly_seq_scheme` 1-based sequence to match frozen FASTA. Verify `auth_seq_num` and residue identity at each observed position using `_atom_site` model 1, and compare native PDB C3′ coordinates with raw mmCIF to **0.02 Å tolerance** (PDB precision).

Only canonical or confidently mapped modified residues are accepted for nucleotide identity. Ambiguous extra chains, nonstandard residues, missing atoms, or mismatches become `REVIEW`/`FAIL`. Molecular provenance and training-data leakage must be investigated separately.

In [ ]:
report=[]
source_hashes={}
for target in TARGETS:
    pdb_id,chain=target.split('_',1)
    cif_path=CIFS/f'{pdb_id}.cif'
    fasta=NUFOLD/'inputs'/target/f'{target}.fasta'
    map_path=NATIVE/f'{target}_residue_map.csv'
    native=NATIVE/f'{target}_native_eval.pdb'
    assert all(p.is_file() for p in (cif_path,fasta,map_path,native)),f'{target}: source file missing'
    seq=read_fasta(fasta)
    residue_map=rows(map_path)
    errors=[];review=[]
    source_hashes[target]={name:hsh(path) for name,path in [('cif',cif_path),('fasta',fasta),('map',map_path),('native_pdb',native)]}
    if len(residue_map)!=len(seq):errors.append('map_length_vs_fasta')
    positions=[int(x['target_position']) for x in residue_map]
    if positions!=list(range(1,len(seq)+1)):errors.append('map_positions_not_exact_1_to_N')
    if any(clean(x.get('pdb_id')).lower()!=pdb_id or clean(x.get('chain_id'))!=chain for x in residue_map):errors.append('map_target_chain_identity')
    yes=[x for x in residue_map if clean(x.get('coordinates_present')).lower()=='yes']
    no=[x for x in residue_map if clean(x.get('coordinates_present')).lower()=='no']
    if len(yes)+len(no)!=len(residue_map):errors.append('unrecognized_coordinate_flag')
    expected_pos={int(x['target_position']) for x in yes}
    block=gemmi.cif.read(str(cif_path)).sole_block()
    if first(block,'_entry.id').lower()!=pdb_id:errors.append('cif_entry_id_mismatch')
    scheme=cif_records(block,'_pdbx_poly_seq_scheme.')
    chain_scheme=[x for x in scheme if clean(x.get('pdb_strand_id'))==chain]
    label_ids=sorted({clean(x.get('asym_id')) for x in chain_scheme})
    if not chain_scheme:errors.append('no_polymer_with_author_chain_id')
    if len(label_ids)!=1:errors.append('author_chain_not_unique_polymer')
    chain_scheme=sorted(chain_scheme,key=lambda r:int(r['seq_id'])) if chain_scheme else []
    declared=''.join(NT_CODES.get(clean(r.get('mon_id')).upper(),'N') for r in chain_scheme)
    if len(chain_scheme)!=len(seq):errors.append('cif_polymer_length_vs_fasta')
    if declared!=seq:errors.append('cif_polymer_sequence_vs_fasta')
    if any(int(x['seq_id'])!=i for i,x in enumerate(chain_scheme,1)):errors.append('cif_polymer_position_numbering')
    if 'N' in declared:review.append('noncanonical_or_unrecognized_polymer_monomer')
    if len(label_ids)==1:
        label=label_ids[0]
    else:label=''

    # Raw mmCIF atom records: model 1, selected author and label chain only.
    atoms=cif_records(block,'_atom_site.')
    atom_map=defaultdict(list)
    chain_models=set()
    for at in atoms:
        if clean(at.get('auth_asym_id'))!=chain:continue
        if label and clean(at.get('label_asym_id'))!=label:continue
        if null(at.get('label_seq_id')):continue
        m=clean(at.get('pdbx_PDB_model_num')) or '1'
        chain_models.add(m)
        if m!='1':continue
        try:p=int(at['label_seq_id'])
        except (TypeError,ValueError):continue
        atom_map[p].append(at)
    atom_observed=set(atom_map)
    if atom_observed!=expected_pos:
        errors.append('coordinate_presence_mismatch_cif_vs_map')
    for x in yes:
        p=int(x['target_position'])
        ats=atom_map.get(p,[])
        if not ats:continue
        if any(NT_CODES.get(clean(a.get('label_comp_id')).upper(),'N')!=seq[p-1] for a in ats):
            errors.append(f'atom_nucleotide_identity_at_{p}')
        auths={clean(a.get('auth_seq_id')) for a in ats if not null(a.get('auth_seq_id'))}
        if len(auths)!=1:review.append(f'ambiguous_atom_author_number_at_{p}')
        elif not number_equal(next(iter(auths)),x.get('auth_seq_num')):errors.append(f'author_seq_num_at_{p}')
        if NT_CODES.get(clean(x.get('pdb_mon_id')).upper(),'N')!=seq[p-1]:errors.append(f'map_pdb_mon_id_at_{p}')
        if p<=len(chain_scheme):
            scheme_auth=clean(chain_scheme[p-1].get('auth_seq_num'))
            # Some deposited mmCIF files do not fill this for unobserved positions.
            if scheme_auth and not number_equal(scheme_auth,x.get('auth_seq_num')):
                errors.append(f'scheme_author_number_at_{p}')
    for x in no:
        p=int(x['target_position'])
        if p in atom_observed:errors.append(f'missing_map_but_atoms_at_{p}')

    pdb_xyz,duplicates=atom_coords_pdb(native)
    if duplicates:errors.append('duplicate_C3prime_native_positions')
    if set(pdb_xyz)!=expected_pos:
        errors.append('native_pdb_C3prime_presence_vs_map')
    mismatch_d=[];missing_c3=[]
    for p in sorted(expected_pos):
        if p not in pdb_xyz:continue
        cand=[a for a in atom_map.get(p,[]) if clean(a.get('label_atom_id'))=="C3'"
              and clean(a.get('label_alt_id')) in ('','A','1')]
        if not cand:
            missing_c3.append(p);continue
        # Alternate conformer: first accepted atom or the highest occupancy.
        cand.sort(key=lambda a:float(a.get('occupancy') or 0) if clean(a.get('occupancy')) else 0,reverse=True)
        a=cand[0]
        try:coord=[float(a[z]) for z in ('Cartn_x','Cartn_y','Cartn_z')]
        except (ValueError,KeyError):errors.append(f'bad_cif_C3prime_at_{p}');continue
        mismatch_d.append((p,distance(coord,pdb_xyz[p])))
    if missing_c3:errors.append('raw_mmcif_missing_C3prime_at_some_mapped_positions')
    max_dev=max((d for _,d in mismatch_d),default=None)
    if max_dev is None or max_dev>0.02:errors.append('native_pdb_coordinate_vs_mmcif_difference')
    if len(chain_models)>1:review.append('multiple_models_found_model1_selected')
    if len(yes)<0.8*len(seq):review.append('native_observed_fraction_below_80pct')

    depdate=first(block,'_pdbx_database_status.recvd_initial_deposition_date')
    revisions=cif_records(block,'_pdbx_audit_revision_history.')
    revision_dates=sorted([clean(x.get('revision_date')) for x in revisions if not null(x.get('revision_date'))])
    if not depdate:review.append('deposition_date_unavailable')
    if not revision_dates:review.append('revision_history_dates_unavailable')
    if not atom_observed:errors.append('no_matching_author_chain_coordinate_atoms')

    status='FAIL' if errors else ('REVIEW' if review else 'PASS')
    report.append({'target_id':target,'pdb_id':pdb_id,'author_chain':chain,'label_asym_id':label,
      'status':status,'fasta_length':len(seq),'cif_polymer_length':len(chain_scheme),
      'native_residue_count':len(expected_pos),'native_fraction':round(len(expected_pos)/len(seq),6),
      'mmcif_atom_observed_count':len(atom_observed),'native_pdb_c3_count':len(pdb_xyz),
      'c3_matches_tested':len(mismatch_d),'max_native_c3_coordinate_difference_angstrom':max_dev,
      'model_numbers':';'.join(sorted(chain_models)),'initial_deposition_date':depdate,
      'earliest_revision_date':revision_dates[0] if revision_dates else '',
      'all_issues':';'.join(dict.fromkeys(errors)),'review_flags':';'.join(dict.fromkeys(review)),
      'cif_sha256':source_hashes[target]['cif'],'native_pdb_sha256':source_hashes[target]['native_pdb'],
      'map_sha256':source_hashes[target]['map'],'fasta_sha256':source_hashes[target]['fasta']})
    print(f'{target:9s} {status:6s} {len(yes):2d}/{len(seq):2d} resolved; '
          f'chain {chain}->{label}; max C3 diff={max_dev}; '
          f'errors={len(errors)} review={len(review)}',flush=True)
assert len(report)==10
print('SUMMARY:',dict(Counter(r['status'] for r in report)))

## 6. Review flags and release-date evidence

A `PASS` means these local checks succeeded—not that a target is free of training leakage or template contamination. `REVIEW` includes incomplete native residues. `FAIL` requires investigation; the notebook still records its audit evidence so problems can be diagnosed without losing results.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns',None)
df=pd.DataFrame(report)
display(df[['target_id','status','author_chain','label_asym_id','fasta_length',
            'native_residue_count','native_fraction','c3_matches_tested',
            'max_native_c3_coordinate_difference_angstrom',
            'initial_deposition_date','earliest_revision_date','all_issues','review_flags']])
print('Never infer no-leakage merely from deposition dates; model training cutoffs are needed.')

## 7. Save NEW immutable audit output and verify checksums

In [ ]:
assert not DEST.exists(),'Refusing to overwrite existing Drive audit'
assert not STAGE.exists(),'Local stage already exists; inspect before rerun'
STAGE.mkdir(parents=True)
output=STAGE/'native_chain_audit.csv'
with output.open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(report[0]),lineterminator='\n')
    writer.writeheader();writer.writerows(report)
counts=dict(Counter(r['status'] for r in report))
summary={
    'audit_id':AUDIT_ID,'created_utc':datetime.now(timezone.utc).isoformat(),
    'status':'complete_with_flags' if counts.get('FAIL',0) or counts.get('REVIEW',0) else 'complete_all_pass',
    'targets':len(report),'counts':counts,'source_base':'MyDrive/RNA3D_Benchmark',
    'input_roots':{'raw_mmcif':'native/pilot_v1/raw_cif','native_pdb_and_maps':'native/pilot_v1/native_eval',
                   'frozen_fasta':'NuFold/runs/pilot10_single_20261018T203832Z/inputs'},
    'method':'mmCIF _pdbx_poly_seq_scheme (author and label chains) + _atom_site model1 + frozen FASTA + native PDB C3prime coordinate check',
    'c3_coordinate_tolerance_angstrom':0.02,
    'source_sha256_by_target':source_hashes,
    'caveats':['No new inference or TM-score recalculation.',
        'A successful structural identity audit is not proof of structural template scarcity.',
        'Depositions and release timestamps alone do not establish absence from model training.',
        'Native experimental C3 coordinates may represent only a subset of each full sequence.',
        'Coordinate checks use C3prime and do not prove all atoms / chemical conformers match.',
        'Any FAIL or REVIEW must be resolved before claiming high-confidence native provenance.']
}
(STAGE/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
DEST.parent.mkdir(exist_ok=True,parents=True)
shutil.copytree(STAGE,DEST)
files=['native_chain_audit.csv','summary.json']
for name in files:assert hsh(STAGE/name)==hsh(DEST/name)
(DEST/'FINALIZED.json').write_text(json.dumps({
    'audit_id':AUDIT_ID,'status':'complete','counts':counts,
    'files':{x:hsh(DEST/x) for x in files},
    'created_utc':datetime.now(timezone.utc).isoformat()
},indent=2)+'\n')
assert (DEST/'FINALIZED.json').is_file()
print('AUDIT SAVED AND VERIFIED:',DEST)
print('TARGETS:',len(report),'PASS:',counts.get('PASS',0),'REVIEW:',counts.get('REVIEW',0),'FAIL:',counts.get('FAIL',0))
print('NO GPU / NO INFERENCE / NO GITHUB WRITE')

## 8. Decision gate (not an automatic publication approval)

- **Any `FAIL`:** inspect mismatch types and chain alignment before accepting results; review whether mapping, chosen author chain, altloc/model selection, or source native PDB needs correction. Preserve original files, use a **new** evaluation version if corrections are needed.
- **Any `REVIEW`:** document incomplete native residue coverage, deposition chronology, and multi-model issues explicitly. Mark samples with <80% native coordinates for sensitivity analysis.
- **All `PASS`:** local chain/residue/C3 provenance passes; **training-data leakage and template scarcity remain unverified** and require independent audits.
- **Next:** look up each model’s training cutoffs, compare deposit/release dates, then perform sequence/structural template screening without leaking experimental coordinates into inference.